# Session 11 · Exercises

*Course material, licence CC BY 4.0.*

Work in pairs. The lecturer chooses which exercises to set; there are more than fit into 30 minutes. *Paper* and *Predict* tasks are answered in the markdown cell below the task, before any code is run; *Code* tasks in the empty code cells. Exercises marked *Stretch* are for pairs who finish early.

## Setup

Run these cells once before the exercises that use data.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# The Berlin listings with the coordinates and the six clustering features of the case study, part A
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
lat0, lon0 = 52.5219, 13.4132                                   # Alexanderplatz
listings["north_km"] = (listings["latitude"] - lat0) * 111.2
listings["east_km"] = (listings["longitude"] - lon0) * 111.2 * np.cos(np.radians(lat0))
listings["km_centre"] = np.hypot(listings["north_km"], listings["east_km"])


def six_features(table):
    """The six numbers per listing used for clustering (log scale for the skewed ones)."""
    return pd.DataFrame({
        "log_price": np.log(table["price"]), "guests": table["accommodates"],
        "log_min_nights": np.log(table["minimum_nights"]), "availability_365": table["availability_365"],
        "log_reviews_ltm": np.log1p(table["number_of_reviews_ltm"]), "km_centre": table["km_centre"],
    })


d = listings.dropna(subset=["price", "minimum_nights"]).reset_index(drop=True)
features = six_features(d)
Z = StandardScaler().fit_transform(features)
print(len(d), "listings with a price;", Z.shape[1], "standardised features in Z")

## Part 1 · Clustering

### 1.1 · k-means with a bad start

*Paper, 15 min*

> Points 1, 2, 3, 10, 11, 25 and k = 2. Run k-means by hand from the starting centroids 1 and 2, and again from the starting centroids 1 and 25. Write down the final clusters and the sum of squared errors (SSE) of each run. Which run would scikit-learn keep, and why does `KMeans` have the parameter `n_init`?

*Source:* original exercise for this course; background: [scikit-learn: K-means (`n_init`)](https://scikit-learn.org/stable/modules/clustering.html)

Your answer:

### 1.2 · Human k-means

*Class, 10 min*

> Two students act as centroids in the room; everyone walks to the nearer one; the centroids move to the middle of their group; repeat until nobody changes sides.

*Source:* original exercise for this course; background: [k-means clustering, Wikipedia](https://en.wikipedia.org/wiki/K-means_clustering)

Your answer:

### 1.3 · Age in years, income in euros

*Paper, 5 min*

> Three customers:
>
> | Customer | Age (years) | Income (€ per year) |
> |---|---|---|
> | A | 25 | 40,000 |
> | B | 62 | 41,000 |
> | C | 27 | 46,000 |
>
> Who is A's nearest neighbour if the features are not scaled? And after standardising, if the standard deviations in the customer table are 15 years and €10,000?

*Source:* original exercise for this course; background: [scikit-learn: Standardization](https://scikit-learn.org/stable/modules/preprocessing.html)

Your answer:

### 1.4 · Draw where k-means fails

*Paper, 5 min*

> Draw two datasets on which k-means with the right k still gives the wrong clusters: two moons, and two concentric rings. Draw the clusters k-means finds.

*Source:* [scikit-learn example: Comparing clustering algorithms](https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_comparison.html)

Your answer:

### 1.5 · A family tree of languages

*Paper, pairs, 20 min*

> The words for *two*, *three*, *night*, *water*, *milk* and *house* in eight EU languages, written without accents:
>
> | | two | three | night | water | milk | house |
> |---|---|---|---|---|---|---|
> | German | zwei | drei | nacht | wasser | milch | haus |
> | Dutch | twee | drie | nacht | water | melk | huis |
> | English | two | three | night | water | milk | house |
> | French | deux | trois | nuit | eau | lait | maison |
> | Spanish | dos | tres | noche | agua | leche | casa |
> | Italian | due | tre | notte | acqua | latte | casa |
> | Polish | dwa | trzy | noc | woda | mleko | dom |
> | Czech | dva | tri | noc | voda | mleko | dum |
>
> Each pair computes the edit distance for two or three language pairs: the number of letters to change, insert or delete, divided by the length of the longer word, averaged over the six words. The lecturer collects the results in one table on the board. Then build the dendrogram by hand: merge the closest pair, and repeat with the average distance between groups (average linkage). Which groups appear?

*Source:* original exercise for this course; background: [Lexicostatistics, Wikipedia](https://en.wikipedia.org/wiki/Lexicostatistics); [Levenshtein distance, Wikipedia](https://en.wikipedia.org/wiki/Levenshtein_distance)

Your answer:

### 1.6 · Segments of the entire homes

*Code, 15 min*

> The worked example clusters all listings, and room type then explains much of the result. Cluster only the entire homes (`room_type == "Entire home/apt"`) with at most 30 minimum nights: build the six features with `six_features`, standardise them, and run k-means for k = 2 to 6 (silhouette with `sample_size=5000`). Choose k, build a profile table with medians on the original scale, and name the segments. Which segments appear now that did not appear before?

*Source:* original exercise for this course, from the exercises of the case study; background: [scikit-learn: Clustering](https://scikit-learn.org/stable/modules/clustering.html)

### 1.7 · Hot spots with HDBSCAN

*Code, 15 min*

> The worked example finds hot spots of listings on the map with `DBSCAN(eps=0.3, min_samples=30)` on the coordinates `d[["east_km", "north_km"]]`. Run `sklearn.cluster.HDBSCAN(min_cluster_size=50)` on the same coordinates. Compare the two: number of hot spots, share of listings marked as noise, and the largest hot spots on a map. Where do they differ, and why?

*Source:* original exercise for this course, from the exercises of the case study; background: [scikit-learn example: Demo of HDBSCAN](https://scikit-learn.org/stable/auto_examples/cluster/plot_hdbscan.html)

### 1.8 · Registration numbers per cluster

*Code, 10 min, Stretch*

> Fit `KMeans(5, n_init=10, random_state=0)` on `Z` and compute, per cluster, the number of listings and the share with a registration number (`d["license"]` not missing). Do the same per DBSCAN hot spot of the previous task. Where would a city office look first? Remember that some values in `license` are placeholders.

*Source:* original exercise for this course, from the exercises of the case study; background: [Inside Airbnb: Data assumptions](https://insideairbnb.com/data-assumptions/)

## Part 2 · Dimensionality reduction and anomaly detection

### 2.1 · Height in cm and in inches

*Paper, 5 min*

> A table has two columns: the height of each person in centimetres and the same height in inches. How many principal components are needed to describe it?

*Source:* original exercise for this course; background: [Principal component analysis, Wikipedia](https://en.wikipedia.org/wiki/Principal_component_analysis)

Your answer:

### 2.2 · The empty middle of a hypercube

*Paper, 5 min*

> In 100 dimensions, what share of a unit cube lies within 0.05 of a face? What does this mean for distances between points in high dimensions?

*Source:* [Curse of dimensionality, Wikipedia](https://en.wikipedia.org/wiki/Curse_of_dimensionality)

Your answer:

### 2.3 · Isolate the outlier

*Class, 10 min*

> The lecturer draws a line of points on the board, one of them far from the others. Students take turns making random cuts between the smallest and the largest point. How many cuts does it take to isolate the extreme point, and how many to isolate a point in the dense middle?

*Source:* [Liu, Ting & Zhou (2008), "Isolation forest", ICDM](https://doi.org/10.1109/ICDM.2008.17)

Your answer:

### 2.4 · PCA of the six listing features

*Code, 15 min*

> Run `PCA()` on the standardised six features `Z`.
>
> 1. How much variance do the components explain, and how many do you need for 80 %?
> 2. Show the loadings of PC1 and PC2 as a table with the feature names. Give each component a name.
> 3. Plot the listings on PC1 and PC2, coloured by room type. Which room types separate?

*Source:* original exercise for this course; background: [scikit-learn: Principal component analysis](https://scikit-learn.org/stable/modules/decomposition.html#pca)

### 2.5 · How stable is a LOF ranking?

*Code, 10 min*

> Take the short-stay listings (`listings["price"].notna() & (listings["minimum_nights"] < 90)`) and standardise four columns: log price, guests, log minimum nights and `km_centre`. Rank the listings with `LocalOutlierFactor` for `n_neighbors` = 5, 20 and 100 (score: `-negative_outlier_factor_`). How many of the top 20 listings do the three rankings share? What does that mean for a list sent to a city office?

*Source:* original exercise for this course; background: [Breunig et al. (2000), "LOF: identifying density-based local outliers", SIGMOD](https://doi.org/10.1145/342009.335388)

### 2.6 · Do location clusters help a linear model?

*Code, pairs, 20 min*

> Part C.2 of the case study shows that 20 k-means location clusters lower the error of a gradient-boosting price model only a little, and raw coordinates do as well. Repeat the comparison with a linear model: one-hot encode room type and district, keep guests, bedrooms and minimum nights, impute missing values, and fit `Ridge` on the log price of the short-stay listings, with `GroupKFold(5)` grouped by `host_id`. Compare three variants: no location, plus the distances to 20 k-means location clusters (fitted inside the pipeline), plus raw coordinates. Do the clusters help more here, and why?

*Source:* original exercise for this course, from the questions of the case study; background: [scikit-learn: Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)

### 2.7 · Amenities as a seventh clustering feature

*Code, 15 min, Stretch*

> Build the amenity score PC1 as in part B.1 of the case study (one 0/1 column per amenity with `MultiLabelBinarizer`, keep amenities offered by 5–95 % of the listings, `PCA`) and add it to the six features as a seventh column. How do the k-means clusters change? Which choice of features answers the question "what kinds of offers exist?" better?

*Source:* original exercise for this course, from the exercises of the case study